# SHARK / Shadow — Memecoin Forward Research Intake
**Cohort:** six owner-supplied Solana mint addresses, enrolled October 12, 2026.

**What this does:** Retrieves live DEX Screener pair data, keeps the exact response and SHA-256, writes a new immutable-named JSON snapshot to the existing Google Drive SHADOW market-replay archive, then independently downloads the file to verify its bytes. No wallet, trading, or transaction signing.

**Important:** This is a market snapshot, not proof of a predictive model. Rerun over time to obtain real point-in-time observations; later grading must match exact token + pair + future horizon. Absence of a pair is not proof the mint does not exist. No historical Shadow ledger is restored here.

**How to run:** Sign in to the same Google account that owns the `SHADOW-PAPER-TRADING/02-MARKET-REPLAY` folder. Run the code cell and authorize the Google Drive permission prompt. The notebook prints a receipt only after readback hash verification. Colab may stop when disconnected, and Google AI subscriptions do not grant indefinite background execution on every tier.

**Note:** the seventh user-supplied address begins with 0x and has an unverified EVM chain. This notebook must NOT send it to Solana endpoints. Telegram mentions for that address remain unscored pending chain verification.


In [ ]:
# SHARK memecoin intake: read-only, no wallet, no orders.
import io, json, hashlib, urllib.request
from datetime import datetime, timezone
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseUpload, MediaIoBaseDownload

MINTS = [
  "7jNwX4Me9zGEjmWY7zARDG2LYjVs1WXD6DZ1JvSfpump",
  "CQCDTp8rsFV3PhaPeYjcCW5LdGTxi6jFx7eC3UXnpump",
  "Fak3rJEBfR4Vb8DoFpecf6yoxbkkr1gGUH5tdj2rpump",  "FXynyTLxTwaLZJrBuLTNafBfqBthbAxKiv7afABapump",
  "9HMuBjwKh3ZemM5k6EC43JiqYyMhaGLeTvuYycfUpump",
  "9Twd5vXAPYSWwuPTVPgDmdxvVHsUQwijC4a1QvYZpump",
]
# Verified from the owner's Drive SHADOW-PAPER-TRADING archive.
PARENT_FOLDER_ID = "1TLZNs6EnLZqsMhP0s-TdHMkQw7hWsoMF"
SUBFOLDER_NAME = "SHARK-MEMECOIN-FORWARD-TRAINING"
CHAIN = "solana"

def utc_now():
    return datetime.now(timezone.utc).isoformat(timespec="microseconds").replace("+00:00", "Z")

def get_source_snapshot():
    endpoint = "https://api.dexscreener.com/tokens/v1/solana/" + ",".join(MINTS)
    request = urllib.request.Request(endpoint, headers={
        "Accept": "application/json", "User-Agent": "Jhadina-SHARK-Research/1.0"
    })
    with urllib.request.urlopen(request, timeout=30) as response:
        if response.status != 200:
            raise RuntimeError("Market provider returned HTTP " + str(response.status))
        raw = response.read()
        if len(raw) > 5_000_000:
            raise RuntimeError("Market response unexpectedly large")
    collected_at = utc_now()
    data = json.loads(raw)
    if not isinstance(data, list):
        raise RuntimeError("Unexpected DEX Screener response schema; nothing saved")
    result = []
    for mint in MINTS:
        matches = []
        for item in data:
            if not isinstance(item, dict) or item.get("chainId") != CHAIN:
                continue
            # DEX Screener priceUsd is for baseToken; quote matches are not price labels.
            base = (item.get("baseToken") or {}).get("address")
            if base != mint:
                continue
            matches.append({
                "pair_address": item.get("pairAddress"),
                "dex_id": item.get("dexId"),
                "market_url": item.get("url"),
                "base_token": item.get("baseToken"),
                "quote_token": item.get("quoteToken"),
                "price_usd_as_reported": item.get("priceUsd"),
                "liquidity": item.get("liquidity"),
                "volume": item.get("volume"),
                "transactions": item.get("txns"),
                "price_changes": item.get("priceChange"),
                "pair_created_at_ms": item.get("pairCreatedAt"),
                "fdv_usd": item.get("fdv"),
                "market_cap_usd": item.get("marketCap"),
            })
        result.append({
            "chain_id": CHAIN, "token_address": mint,
            "status": "PAIRS_OBSERVED" if matches else "NO_BASE_TOKEN_PAIR_IN_RESPONSE",
            "pairs": matches,
        })
    snapshot = {
        "schema": "shark-memecoin-observation/v1",
        "cohort_date": "2026-10-12",
        "collected_at_utc": collected_at,
        "timestamp_semantics": "Collector received-at only; provider quote tick time not established",
        "provider": "dexscreener",
        "request_url": endpoint,
        "response_sha256": hashlib.sha256(raw).hexdigest(),
        "raw_api_response_utf8": raw.decode("utf-8"),
        "tokens": result,
        "forward_data": True,
        "trade_authority": "NONE",
        "simulated_order": False,
        "prediction": None,
        "outcome": None,
        "scoring_eligible": False,
    }
    return snapshot

def upload_and_verify(service, folder_id, name, payload):
    data = payload.encode("utf-8")
    media = MediaIoBaseUpload(io.BytesIO(data), mimetype="application/json", resumable=False)
    obj = service.files().create(
        body={"name": name, "parents": [folder_id], "mimeType": "application/json"},
        media_body=media, fields="id,name,webViewLink"
    ).execute()
    downloaded = io.BytesIO()
    req = service.files().get_media(fileId=obj["id"])
    downloader = MediaIoBaseDownload(downloaded, req)
    done = False
    while not done:
        _, done = downloader.next_chunk()
    if hashlib.sha256(downloaded.getvalue()).digest() != hashlib.sha256(data).digest():
        raise RuntimeError("Google Drive readback hash mismatch; do not use snapshot")
    return obj

auth.authenticate_user()
drive = build("drive", "v3")
root = drive.files().get(fileId=PARENT_FOLDER_ID, fields="id,name,mimeType").execute()
if root["mimeType"] != "application/vnd.google-apps.folder":
    raise RuntimeError("Unexpected SHADOW market replay destination")
query = "name = '%s' and '%s' in parents and trashed = false" % (SUBFOLDER_NAME, PARENT_FOLDER_ID)
existing = drive.files().list(q=query, fields="files(id,name,mimeType)", page_size=100).execute()["files"]
folders = [x for x in existing if x["mimeType"] == "application/vnd.google-apps.folder"]
if len(folders) > 1:
    raise RuntimeError("Multiple identically named folders: review manually")
folder = folders[0] if folders else drive.files().create(
    body={"name": SUBFOLDER_NAME, "mimeType": "application/vnd.google-apps.folder",
          "parents": [PARENT_FOLDER_ID]}, fields="id,name,mimeType"
).execute()
snapshot = get_source_snapshot()
payload = json.dumps(snapshot, ensure_ascii=False, sort_keys=True, indent=2) + "\n"
filename = "SHARK_MEME_SNAPSHOT_" + snapshot["collected_at_utc"].replace(":", "-") + ".json"
saved = upload_and_verify(drive, folder["id"], filename, payload)
print("VERIFIED_GOOGLE_DRIVE_READBACK", saved["id"])
print("UTC timestamp:", snapshot["collected_at_utc"])
print("Source response SHA-256:", snapshot["response_sha256"])
for token in snapshot["tokens"]:
    print(token["token_address"], token["status"], "pairs:", len(token["pairs"]))
print("Research-only snapshot; no predictions or profitable-trading claims.")


## Forward labels (optional, only after repeated runs)

This cell reads previously saved snapshots. It pairs **the same Solana token and pair** with a later snapshot at each target horizon, never using an earlier or mismatched quote. It only creates observational return labels. It does **not** train a predictive model, fabricate fills, or grant live-trading authority.


In [ ]:
# Run this cell only AFTER collecting more than one snapshot.
# Creates forward observed-return training labels, NOT graded model predictions.
from collections import defaultdict
from datetime import timedelta
from decimal import Decimal, InvalidOperation

HORIZONS = [(15, 5), (60, 12), (240, 30), (1440, 60), (4320, 120), (10080, 180)]
def read_snapshot(file_id):
    buffer = io.BytesIO()
    downloader = MediaIoBaseDownload(buffer, drive.files().get_media(fileId=file_id))
    done = False
    while not done:
        _, done = downloader.next_chunk()
    obj = json.loads(buffer.getvalue())
    if obj.get("schema") != "shark-memecoin-observation/v1":
        raise ValueError("Unexpected snapshot schema")
    raw = obj.get("raw_api_response_utf8", "").encode("utf-8")
    if hashlib.sha256(raw).hexdigest() != obj.get("response_sha256"):
        raise ValueError("Provider response hash mismatch")
    if obj.get("provider") != "dexscreener":
        raise ValueError("Unexpected data provider")
    return obj

snapfiles, page = [], None
while True:
    response = drive.files().list(
        q="'%s' in parents and trashed = false" % folder["id"],
        fields="nextPageToken,files(id,name,mimeType)",
        page_size=100, pageToken=page).execute()
    snapfiles += [f for f in response.get("files", [])
                  if f["name"].startswith("SHARK_MEME_SNAPSHOT_")
                  and f["name"].endswith(".json")]
    if len(snapfiles) > 500:
        raise RuntimeError("Over 500 snapshots: process in batches instead")
    page = response.get("nextPageToken")
    if not page:
        break

series = defaultdict(list)
for f in snapfiles:
    s = read_snapshot(f["id"])
    t = datetime.fromisoformat(s["collected_at_utc"].replace("Z", "+00:00"))
    for token in s["tokens"]:
        mint = token["token_address"]
        if mint not in MINTS:
            continue
        for pair in token["pairs"]:
            if (pair.get("base_token") or {}).get("address") != mint:
                continue
            pairid = pair.get("pair_address")
            if not pairid:
                continue
            try:
                price = Decimal(str(pair.get("price_usd_as_reported")))
            except (InvalidOperation, TypeError, ValueError):
                continue
            if not price.is_finite() or price <= 0:
                continue
            series[(mint, pairid)].append((t, price, f["id"]))

labels = []
for (mint, pairid), samples in series.items():
    samples.sort(key=lambda row: row[0])
    for start_time, first_price, first_id in samples:
        for minutes, tolerance in HORIZONS:
            lower = start_time + timedelta(minutes=minutes)
            upper = lower + timedelta(minutes=tolerance)
            matches = [(t, p, source_id) for t, p, source_id in samples
                       if lower <= t <= upper and source_id != first_id]
            if not matches:
                continue  # Missing timing coverage is NOT a negative outcome.
            observed_at, end_price, end_id = min(matches, key=lambda row: row[0])
            pct = (end_price / first_price - Decimal(1)) * Decimal(100)
            labels.append({
                "chain_id": "solana", "token_address": mint,
                "pair_address": pairid,
                "baseline_file_id": first_id, "outcome_file_id": end_id,
                "baseline_observed_at_utc": start_time.isoformat(),
                "outcome_observed_at_utc": observed_at.isoformat(),
                "horizon_minutes": minutes,
                "return_pct_observed": str(pct.quantize(Decimal("0.000001"))),
                "direction": "UP" if pct > 0 else ("DOWN" if pct < 0 else "FLAT"),
                "outcome_kind": "SNAPSHOT_PRICE_RETURN_ONLY",
                "includes_fees_slippage_or_execution": False,
                "forecast_skill_established": False,
            })

print("Snapshots:", len(snapfiles), "usable token/pair series:", len(series))
print("Strictly time-aligned observed-return candidates:", len(labels))
if labels:
    labels_doc = {
        "schema": "shark-memecoin-forward-labels/v1",
        "created_at_utc": utc_now(),
        "derived_from": "validated_collector_snapshots",
        "notes": "Observed return labels are NOT proof of executable trade profits or predictive skill.",
        "labels": labels
    }
    name = "SHARK_MEME_LABELS_" + labels_doc["created_at_utc"].replace(":", "-") + ".json"
    receipt = upload_and_verify(
        drive, folder["id"], name,
        json.dumps(labels_doc, sort_keys=True, indent=2) + "\n"
    )
    print("LABEL_READBACK_VERIFIED", receipt["id"])
else:
    print("NO_HORIZON_LABELS_YET: Collect future snapshots; never backfill with today's price.")
